> **Revision note (2026-09-23).** This notebook produced Table 14 of the original submission and is kept unchanged for provenance. Its loader (`CrossDatasetTestDataset`) omits the brain-cropping and median-filtering steps used in training, and it does not account for the viridis pseudo-colour encoding of the public D3 release. Both errors are documented in Section 4.10 of the revised manuscript. The corrected re-evaluation, which first reproduces this notebook's six accuracies exactly, is [`audit/kaggle/01_table14_transfer_audit.py`](audit/kaggle/01_table14_transfer_audit.py); see [`audit/README.md`](audit/README.md).

In [ ]:
# ============================================================
# Q1-Defensible Cross-Dataset Evaluation
# Uses pHash group-aware target test split
# ============================================================

import os
import json
import cv2
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    cohen_kappa_score,
    confusion_matrix,
    classification_report,
)

# ------------------------------------------------------------
# User settings
# ------------------------------------------------------------

SOURCE = "D4"   # "D2" or "D4"
TARGET = "D3"   # "D2", "D3", or "D4"

SOURCE_CKPT = "/kaggle/input/datasets/rogo2019/cross-dataset-assets/cross-dataset-assets/D4/final_model_v5_calibrated_pytorch.pth"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMG_SIZE = 224
BATCH_SIZE = 32

# ------------------------------------------------------------
# Dataset configuration
# ------------------------------------------------------------

DATASET_CONFIGS = {
    "D2": {
        "name": "BRISC-2025",
        "num_classes": 4,
        "class_labels": ["glioma", "meningioma", "no_tumor", "pituitary"],
        "split_csv": "/kaggle/input/datasets/rogo2019/cross-dataset-assets/cross-dataset-assets/D2/D2_phash_group_split.csv",
    },

    "D3": {
        "name": "Figshare (Cheng 2017)",
        "num_classes": 3,
        "class_labels": ["glioma", "meningioma", "pituitary"],
        "split_csv": "/kaggle/input/datasets/rogo2019/cross-dataset-assets/cross-dataset-assets/D3/D3_phash_group_split.csv",
    },

    "D4": {
        "name": "Nickparvar Combined",
        "num_classes": 4,
        "class_labels": ["glioma", "meningioma", "no_tumor", "pituitary"],
        "split_csv": "/kaggle/input/datasets/rogo2019/cross-dataset-assets/cross-dataset-assets/D4/D4_phash_group_split.csv",
    },
}

assert SOURCE in DATASET_CONFIGS, f"Unknown SOURCE={SOURCE}"
assert TARGET in DATASET_CONFIGS, f"Unknown TARGET={TARGET}"
assert os.path.exists(SOURCE_CKPT), f"Checkpoint not found: {SOURCE_CKPT}"

src_cfg = DATASET_CONFIGS[SOURCE]
tgt_cfg = DATASET_CONFIGS[TARGET]

assert os.path.exists(tgt_cfg["split_csv"]), (
    f"Target split CSV not found: {tgt_cfg['split_csv']}\n"
    "Run the pHash group-aware split first."
)

print("=" * 70)
print("Q1 CROSS-DATASET ZERO-SHOT EVALUATION")
print("=" * 70)
print(f"Source model : {SOURCE} ({src_cfg['name']})")
print(f"Target test  : {TARGET} ({tgt_cfg['name']})")
print(f"Checkpoint   : {SOURCE_CKPT}")
print("=" * 70)

# ------------------------------------------------------------
# Load target pHash-safe test split
# ------------------------------------------------------------

df_split = pd.read_csv(tgt_cfg["split_csv"])

if "split" not in df_split.columns:
    raise ValueError("Split CSV must contain a 'split' column.")

if "path" not in df_split.columns:
    raise ValueError("Split CSV must contain a 'path' column.")

if "label" not in df_split.columns:
    raise ValueError("Split CSV must contain a 'label' column.")

df_test = df_split[df_split["split"] == "test"].reset_index(drop=True)

print(f"\n[LOAD] Target test samples from pHash-safe split: {len(df_test)}")
print(df_test["label"].value_counts().to_dict())

# ------------------------------------------------------------
# Shared-class handling
# ------------------------------------------------------------

src_labels = src_cfg["class_labels"]
tgt_labels = tgt_cfg["class_labels"]

shared_labels = [lbl for lbl in tgt_labels if lbl in src_labels]
shared_indices_in_source = [src_labels.index(lbl) for lbl in shared_labels]

if len(shared_labels) < len(tgt_labels):
    print("\n[INFO] Restricted shared-class evaluation")
    print(f"Target labels      : {tgt_labels}")
    print(f"Shared labels      : {shared_labels}")
    print(f"Excluded labels    : {[x for x in tgt_labels if x not in shared_labels]}")

df_test = df_test[df_test["label"].isin(shared_labels)].reset_index(drop=True)

label_to_idx = {label: i for i, label in enumerate(shared_labels)}
y_true = np.array([label_to_idx[x] for x in df_test["label"]], dtype=np.int64)

print(f"\n[EVAL] Final evaluated test samples: {len(df_test)}")
print(f"[EVAL] Evaluated classes: {shared_labels}")
print(f"[EVAL] Per-class support: {dict(zip(shared_labels, np.bincount(y_true)))}")

# ------------------------------------------------------------
# Dataset class
# ------------------------------------------------------------

class CrossDatasetTestDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, labels):
        self.df = dataframe.reset_index(drop=True)
        self.labels = labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.df.loc[idx, "path"]

        img = cv2.imread(path)
        if img is None:
            raise RuntimeError(f"Could not read image: {path}")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)

        img = img.astype(np.float32) / 255.0

        # ImageNet normalization, same as timm/torchvision pretrained models
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img = (img - mean) / std

        img = np.transpose(img, (2, 0, 1))
        img = torch.tensor(img, dtype=torch.float32)

        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, y


test_ds = CrossDatasetTestDataset(df_test, y_true)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

# ------------------------------------------------------------
# Load source model
# ------------------------------------------------------------

print("\n[MODEL] Loading source calibrated model...")

ckpt = torch.load(SOURCE_CKPT, map_location=DEVICE)

temperature = float(ckpt.get("temperature", 1.0))

state_dict = ckpt["model_state_dict"]
state_dict = {
    (k[7:] if k.startswith("module.") else k): v
    for k, v in state_dict.items()
    if k != "n_averaged"
}

model = BrainTumorModel(num_classes=src_cfg["num_classes"]).to(DEVICE)
model.load_state_dict(state_dict, strict=True)
model.eval()

print(f"[MODEL] Loaded source model with {src_cfg['num_classes']} classes")
print(f"[MODEL] Temperature T = {temperature:.4f}")

# ------------------------------------------------------------
# Inference
# ------------------------------------------------------------

all_logits = []
all_targets = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)

        if hasattr(model, "backbone_forward"):
            logits = model.backbone_forward(xb)
        else:
            logits = model(xb)

        all_logits.append(logits.cpu())
        all_targets.append(yb.cpu())

all_logits = torch.cat(all_logits, dim=0)
all_targets = torch.cat(all_targets, dim=0).numpy()

# Restrict logits to shared target classes
logits_shared = all_logits[:, shared_indices_in_source]

# Apply source temperature
logits_shared = logits_shared / temperature

probs = torch.softmax(logits_shared, dim=1).numpy()
y_pred = np.argmax(probs, axis=1)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

acc = accuracy_score(all_targets, y_pred)
prec = precision_score(all_targets, y_pred, average="weighted", zero_division=0)
rec = recall_score(all_targets, y_pred, average="weighted", zero_division=0)
f1 = f1_score(all_targets, y_pred, average="weighted", zero_division=0)
kappa = cohen_kappa_score(all_targets, y_pred)

try:
    y_onehot = np.eye(len(shared_labels))[all_targets]
    auc = roc_auc_score(
        y_onehot,
        probs,
        multi_class="ovr",
        average="macro",
    )
except Exception:
    auc = float("nan")

brier = float(
    np.mean(
        np.sum(
            (probs - np.eye(len(shared_labels))[all_targets]) ** 2,
            axis=1,
        )
    )
)

cm = confusion_matrix(all_targets, y_pred)

report = classification_report(
    all_targets,
    y_pred,
    target_names=shared_labels,
    digits=4,
    zero_division=0,
)

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print(f"CROSS-DATASET RESULTS: {SOURCE} → {TARGET}")
print("Zero-shot evaluation; no target fine-tuning")
print("=" * 70)
print(f"Evaluated classes : {shared_labels}")
print(f"n_test            : {len(all_targets)}")
print(f"Accuracy          : {acc * 100:.2f}%")
print(f"Precision         : {prec * 100:.2f}%")
print(f"Recall            : {rec * 100:.2f}%")
print(f"F1-score          : {f1 * 100:.2f}%")
print(f"AUC               : {auc * 100:.2f}%")
print(f"Kappa             : {kappa:.4f}")
print(f"Brier score       : {brier:.4f}")
print("=" * 70)

print("\nClassification report:")
print(report)

print("Confusion matrix:")
print(cm)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

out_dir = f"./outputs_pytorch/cross_dataset"
os.makedirs(out_dir, exist_ok=True)

result = {
    "source": SOURCE,
    "source_name": src_cfg["name"],
    "target": TARGET,
    "target_name": tgt_cfg["name"],
    "evaluation_type": "zero_shot_cross_dataset",
    "target_split_source": tgt_cfg["split_csv"],
    "pHash_safe_target_test_partition": True,
    "class_handling": (
        "shared_class_restricted"
        if len(shared_labels) < len(tgt_labels)
        else "full_class_evaluation"
    ),
    "source_classes": src_labels,
    "target_classes": tgt_labels,
    "evaluated_classes": shared_labels,
    "n_test": int(len(all_targets)),
    "per_class_support": {
        shared_labels[i]: int(c)
        for i, c in enumerate(np.bincount(all_targets))
    },
    "metrics": {
        "accuracy": round(acc * 100, 4),
        "precision": round(prec * 100, 4),
        "recall": round(rec * 100, 4),
        "f1": round(f1 * 100, 4),
        "auc": round(auc * 100, 4) if not np.isnan(auc) else None,
        "kappa": round(kappa, 4),
        "brier": round(brier, 4),
    },
    "temperature_used": temperature,
    "confusion_matrix": cm.tolist(),
    "classification_report": report,
}

out_json = os.path.join(
    out_dir,
    f"cross_dataset_{SOURCE}_to_{TARGET}_phash_safe.json",
)

with open(out_json, "w") as f:
    json.dump(result, f, indent=2)

print(f"\n[SAVED] {out_json}")
